# Módulo 45: Modelado de datos — Guía del ejercicio

En este ejercicio vas a organizar las tablas que limpiaste en el Módulo 44 en un **modelo estrella** con DuckDB: una tabla de hechos de ventas en el centro y cuatro dimensiones relacionadas con ella. Comprobarás las claves y la cardinalidad de cada relación, documentarás el modelo y escribirás consultas que lo atraviesan.

- Si necesitas repasar los conceptos, consulta la lección [m45-modelado-de-datos-leccion.ipynb](m45-modelado-de-datos-leccion.ipynb).
- Cuando termines, compara tu trabajo con la solución: [m45-modelado-de-datos-solucion.ipynb](m45-modelado-de-datos-solucion.ipynb).

## Contexto

Una cadena de tiendas con presencia en diez países de América y Europa ya tiene limpias sus ventas de 2019 a 2021 y sus catálogos de países, clientes, productos y tiendas. Por separado, esas tablas no responden preguntas como "¿cuánto vendió cada categoría en Canadá?" o "¿qué tiendas venden más muebles?".

Antes de construir ningún dashboard, el equipo de análisis necesita **relacionar** las tablas para que un filtro sobre un país, un producto, un cliente o una tienda llegue a las ventas en un solo salto, sin ambigüedades y sin inflar los totales.

## Datos

El modelo limpio está en [`../practica-ventas/datos-modelo/`](../practica-ventas/datos-modelo/) y se carga con `conectar()` de [`../practica-ventas/ventas.py`](../practica-ventas/ventas.py), que crea una base DuckDB en memoria con estas tablas:

| Tabla | Filas | Columnas |
| --- | --- | --- |
| `ventas` | 1 499 000 (2019–2021) | `fecha`, `id_pais`, `id_tienda`, `id_cliente`, `id_producto`, `cantidad`, `precio`, `importe` |
| `paises` | 10 | `id_pais`, `pais` |
| `clientes` | 950 | `id_cliente`, `nombre`, `apellido`, `correo`, `telefono` |
| `productos` | 1000 | `id_producto`, `categoria`, `subcategoria`, `producto` |
| `tiendas` | 200 | `id_tienda`, `tienda`, `direccion`, `formato` |
| `calendario` | 1096 | `fecha`, `anio`, `trimestre`, `mes`, `nombre_mes`, `anio_mes` |

Además crea la vista `ventas_detalle`, que ya une los hechos con sus dimensiones. En este ejercicio **no la uses** salvo para comparar: el objetivo es que escribas tú las uniones.

Para la Tarea 6 necesitarás también la hoja cruda `t_store` de `../practica-ventas/datos-ventas/catalogo.xlsx` (se lee con `header=2`), sin limpiar.

## Instrucciones

- Trabaja en este notebook, ejecutándolo desde su carpeta para que las rutas relativas funcionen (`uv run jupyter lab` desde la raíz del repositorio).
- Escribe las consultas en SQL con DuckDB (`con.sql(...).df()` devuelve un DataFrame de pandas); usa pandas para lo que te resulte más cómodo.
- Después de cada tarea, comprueba el resultado y explica en una o dos frases qué muestra.

La celda siguiente importa las librerías y la práctica de ventas como `practica` (el nombre `ventas` queda libre para tus DataFrames).

In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "png"
pio.defaults.default_width = 900
pio.defaults.default_height = 450

sys.path.insert(0, str(Path("../practica-ventas").resolve()))
import ventas as practica

DATOS = Path("../practica-ventas/datos-ventas")

## Tarea 1. Cargar el modelo y clasificar sus tablas

1. Abre el modelo con `con = practica.conectar()` y lista sus tablas y vistas.
2. Para cada tabla, muestra el número de filas y sus columnas con su tipo (`DESCRIBE`).
3. Clasifica cada tabla como **hechos** o **dimensión** y describe la **granularidad** de la tabla de hechos: ¿qué representa una fila? ¿Hay alguna combinación de columnas que la identifique de forma única?
4. Indica qué columnas de `ventas` son medidas **aditivas** (se pueden sumar por cualquier dimensión) y cuáles no.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 2. Comprobar claves y tipos

1. Para cada dimensión, compara `count(*)` con `count(DISTINCT clave)`: la clave debe tener tantos valores distintos como filas.
2. Compara el tipo de cada clave en la dimensión y en `ventas`. ¿Son compatibles?
3. Cuenta las ventas **huérfanas** (claves de `ventas` que no existen en la dimensión) y las claves de la dimensión **sin ventas**. Usa `ANTI JOIN` o `NOT EXISTS`.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 3. Construir la tabla de relaciones

1. Busca las columnas que `ventas` comparte por nombre con cada una de las otras tablas: son las **relaciones candidatas**, las que una herramienta de BI detectaría automáticamente. Descarta la vista `ventas_detalle`, que no es una tabla del modelo.
2. Decide qué candidatas son relaciones válidas y construye un DataFrame `relaciones` con una fila por relación y las columnas `desde` (lado varios), `hacia` (lado uno), `clave`, `cardinalidad`, `direccion_filtro` y `activa`.
3. Compara tus cuatro relaciones con dimensiones con `practica.validar_modelo(con)`.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 4. Documentar el modelo

1. A partir de `relaciones`, imprime un diagrama de texto de la estrella: `ventas` en el centro y cada dimensión con su clave y los indicadores `1` y `*`.
2. Escribe una tabla resumen con una fila por tabla: tipo (hechos o dimensión), filas, clave y columnas que verá quien construya el dashboard (las descriptivas, no las claves técnicas).

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 5. Consultas que atraviesan las relaciones

1. Calcula las ventas por categoría en Canadá filtrando **desde las dimensiones** (`paises.pais = 'CANADA'`), no con el número de la clave.
2. Obtén las cinco tiendas que más venden muebles (`Furniture`), con su clave, su nombre y su formato. ¿Por qué conviene agrupar por la clave y no solo por el nombre?
3. Dibuja con Plotly un gráfico de barras con el resultado de una de las dos consultas.
4. Comprueba tus resultados con `practica.ventas_por(con, ..., paises=[...], categorias=[...])`.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 6. Por qué la clave de la dimensión debe ser única

1. Lee la hoja cruda `t_store` (204 filas) y regístrala en DuckDB con `con.register("t_store_crudo", ...)`.
2. Comprueba que `ID_Store` no es única y qué tiendas están repetidas.
3. Une `ventas` con `t_store_crudo` y compara el número de filas y las ventas totales con los de `ventas` sola. Explica de dónde sale la diferencia.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Antes de terminar

- [ ] Las claves de las cuatro dimensiones son únicas, de tipo entero y compatibles con las de `ventas`, sin huérfanas.
- [ ] `relaciones` tiene una fila por relación válida, varios a uno, activa y con filtro de la dimensión a los hechos.
- [ ] El diagrama de texto muestra la estrella completa.
- [ ] Las consultas filtran por columnas de las dimensiones y coinciden con `practica.ventas_por`.
- [ ] Explicaste cuánto infla las ventas la hoja `t_store` sin limpiar.

Cuando termines, compara tu trabajo con la solución: [m45-modelado-de-datos-solucion.ipynb](m45-modelado-de-datos-solucion.ipynb).